In [1]:
import gzip
import os
import re
import sys

import gffutils
from Bio import SeqIO
from Bio.Seq import Seq

#import PEGG from the checkout beside this repo, the same way pegviz does
PEGG_PATH = os.environ.get('PEGG_PATH') or os.path.abspath(
    os.path.join(os.getcwd(), '..', 'PEGG3.0'))
if os.path.isdir(os.path.join(PEGG_PATH, 'pegg')):
    sys.path.insert(0, PEGG_PATH)

from pegg import bystander

DB_ROOT = '/Users/kexindong/Documents/GitHub/Database'

#annotation and genome must be the SAME build, or the CDS blocks index the
#wrong bases -- a silent failure that get_exon()'s reference-base check catches
ANNOTATION = {
    ('human', 37): f'{DB_ROOT}/Genecode/gencode_v19_GRCh37.db',
    ('human', 38): f'{DB_ROOT}/Genecode/gencode_v44_GRCh38.db',
    ('mouse', 39): f'{DB_ROOT}/Genecode/gencode_vm33_GRCm39.db',
}
GENOME = {
    ('human', 37): f'{DB_ROOT}/RefGenome/ncbi-2023-09-12/GCF_000001405.25_GRCh37.p13_genomic.fna.gz',
    ('human', 38): f'{DB_ROOT}/RefGenome/ncbi-2023-09-12/GCF_000001405.40_GRCh38.p14_genomic.fna.gz',
    ('mouse', 39): f'{DB_ROOT}/RefGenome/mouse-2023-09-13/GCF_000001635.27_GRCm39_genomic.fna.gz',
}

In [2]:
_CHROM_CACHE = {}


def _chromosome(species, build, chrom):
    """One chromosome as a string. Parses only until it is found, then caches."""
    chrom = str(chrom).replace('chr', '')
    key = (species, build, chrom)
    if key not in _CHROM_CACHE:
        skip = ('alternate', 'unplaced', 'unlocalized', 'patch', 'mitochondrion')
        with gzip.open(GENOME[(species, build)], 'rt') as handle:
            for record in SeqIO.parse(handle, 'fasta'):
                if any(w in record.description for w in skip):
                    continue
                found = re.search(r'chromosome (\w+)', record.description)
                if found and found.group(1) == chrom:
                    _CHROM_CACHE[key] = str(record.seq).upper()
                    break
            else:
                raise ValueError('chromosome %r not found' % chrom)
    return _CHROM_CACHE[key]


#three-letter names appear in HGVS ("p.Arg175His"); accept either spelling
_AA3 = {'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Gln':'Q','Glu':'E',
        'Gly':'G','His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F',
        'Pro':'P','Ser':'S','Thr':'T','Trp':'W','Tyr':'Y','Val':'V','Ter':'*'}


def _resolve_protein_change(call, ann, chrom_seq, verbose=True):
    """Turns 'R175H' into (genomic_position, ref_base, alt_base).

    A protein change does not name a DNA change: several substitutions reach the
    same residue, and most need more than one base changed. Only single-base
    routes are returned, since that is what one pegRNA installs cleanly; if a
    change needs two or three bases, that is reported rather than guessed at.
    """
    text = call.strip()
    if text.lower().startswith('p.'):
        text = text[2:]

    found = re.fullmatch(r'([A-Z](?:[a-z]{2})?)(\d+)([A-Z](?:[a-z]{2})?|\*|=)', text)
    if not found:
        raise ValueError(
            'could not read %r as a protein change. Expected e.g. R175H, '
            'p.Arg175His, or R175* .' % call)

    from_aa, number, to_aa = found.groups()
    from_aa = _AA3.get(from_aa, from_aa)
    to_aa = _AA3.get(to_aa, to_aa)
    number = int(number)

    #'X' and friends parse as letters but are not residues; catching them here
    #gives a straight answer instead of "no codon codes for it"
    known = set('ACDEFGHIKLMNPQRSTVWY*')
    for aa, role in ((from_aa, 'reference'), (to_aa, 'variant')):
        if aa not in known and aa != '=':
            raise ValueError('%r is not an amino acid (%s residue in %r). Use a '
                             'one-letter code, a three-letter code, or * for a '
                             'stop.' % (aa, role, call))

    positions = bystander.cds_positions(ann['cds'], ann['strand'])
    codon_positions = positions[(number - 1) * 3:number * 3]
    if len(codon_positions) < 3:
        raise ValueError('%s has only %d codons; residue %d does not exist'
                         % (ann['transcript_id'], len(positions) // 3, number))

    codon = ''.join(chrom_seq[p - 1] for p in codon_positions)
    if ann['strand'] == '-':
        codon = str(Seq(codon).complement())

    observed = str(Seq(codon).translate())
    if observed != from_aa:
        raise ValueError(
            'residue %d of this transcript is %s, not %s. Check the build and '
            'that the numbering refers to %s.'
            % (number, observed, from_aa, ann['transcript_id']))
    if to_aa == '=':
        to_aa = observed

    #every single-base change of this codon that gives the wanted residue
    routes = []
    for offset in range(3):
        for base in 'ACGT':
            if base == codon[offset]:
                continue
            candidate = codon[:offset] + base + codon[offset + 1:]
            if str(Seq(candidate).translate()) != to_aa:
                continue
            genomic = codon_positions[offset]
            #alleles are quoted on the genome, so flip back for a '-' transcript
            flip = (lambda b: str(Seq(b).reverse_complement())) \
                if ann['strand'] == '-' else (lambda b: b)
            routes.append({'position': genomic, 'codon': candidate,
                           'transcript_ref': codon[offset], 'transcript_alt': base,
                           'ref': flip(codon[offset]), 'alt': flip(base)})

    if not routes:
        need = min(sum(1 for a, b in zip(codon, cand) if a != b)
                   for cand in _codons_for(to_aa))
        raise ValueError(
            '%s%d%s needs %d bases changed from %s, not 1. Prime editing installs '
            'it fine, but this resolver reports a single base -- give the genomic '
            'coordinate and alt allele explicitly instead.'
            % (from_aa, number, to_aa, need, codon))

    chosen = routes[0]
    if verbose:
        print('%s%d%s  ->  codon %d %s>%s at chr%s:%d (%s>%s on the genome)'
              % (from_aa, number, to_aa, number, codon, chosen['codon'],
                 str(ann['chrom']).replace('chr', ''), chosen['position'],
                 chosen['ref'], chosen['alt']))
        if len(routes) > 1:
            others = ', '.join('%s>%s at %d' % (r['ref'], r['alt'], r['position'])
                               for r in routes[1:])
            print('  %d other single-base routes: %s' % (len(routes) - 1, others))
        print()

    return chosen['position'], chosen['ref'], chosen['alt']


def _codons_for(aa):
    """Every codon coding for one amino acid, for the 'needs N changes' message."""
    return [c for c, a in bystander.CODON_TO_AA.items() if a == aa]


def get_exon(gene, variant, ref=None, alt=None,
             species='human', build=37, transcript_id=None, verbose=True):
    """Returns the exon containing a coding variant, ready to paste.

    `variant` is either a genomic coordinate (with ref/alt bases) or a protein
    change in the usual notation::

        get_exon('TP53', 7577121, ref='G', alt='A')    # genomic
        get_exon('TP53', 'R175H')                      # protein

    The transcript is the canonical one (the curated table H2M and PEGG use), so
    the frame matches the transcript your variant annotation was computed
    against -- which is what makes 'R175H' unambiguous. The sequence runs in the
    transcript's direction and is trimmed to whole codons, so it is in frame
    from offset 0.

    Raises ValueError for a non-coding position: with no reading frame there is
    nothing for a bystander to be silent against.
    """

    #the canonical-transcript table is keyed by build for human only; for mouse
    #PEGG ignores the version and rejects anything but 37/38
    tx = transcript_id or bystander.canonical_transcript(
        gene, species=species, genome_version=build if species == 'human' else 37)
    if tx is None:
        raise ValueError('no canonical transcript known for %r; pass '
                         'transcript_id=' % gene)

    ann = bystander.cds_from_annotation_db(
        gffutils.FeatureDB(ANNOTATION[(species, build)]), tx)
    chrom_seq = _chromosome(species, build, ann['chrom'])

    #'R175H' -> a genomic coordinate. Resolved against this transcript, which is
    #the same one the protein numbering refers to, so the two cannot disagree.
    protein_call = None
    if isinstance(variant, str) and not variant.isdigit():
        protein_call = variant
        variant, ref, alt = _resolve_protein_change(
            variant, ann, chrom_seq, verbose=verbose)

    genomic_position = int(variant)

    #which CDS block holds the mutation
    exon = exon_index = None
    for index, (start, end) in enumerate(ann['cds']):
        if start <= genomic_position <= end:
            exon, exon_index = (start, end), index
            break
    if exon is None:
        raise ValueError(
            '%s is not in the CDS of %s. Only coding variants are supported: an '
            'intronic or UTR position has no reading frame for a bystander to be '
            'silent against.' % (genomic_position, tx))

    #index of every coding base along the transcript, from PEGG's own walk. The
    #phase (index % 3) drives the trim, so it respects the frame carried across
    #introns; the index itself gives each codon its number in the full protein.
    cds_index = {p: i for i, p in enumerate(
        bystander.cds_positions(ann['cds'], ann['strand']))}
    phase = {p: i % 3 for p, i in cds_index.items()}

    span = list(range(exon[0], exon[1] + 1))
    if ann['strand'] == '-':
        span = span[::-1]

    #drop the leading bases finishing the previous exon's codon, and any
    #trailing bases that would start the next one
    first = next(i for i, p in enumerate(span) if phase[p] == 0)
    kept = span[first:first + (len(span) - first) // 3 * 3]
    if not kept:
        raise ValueError('exon %d of %s holds no complete codon'
                         % (exon_index + 1, tx))
    if genomic_position not in kept:
        raise ValueError(
            '%s falls in a codon split across the intron, which cannot be edited '
            'from a single RTT.' % genomic_position)

    sequence = ''.join(chrom_seq[p - 1] for p in kept)
    if ann['strand'] == '-':
        sequence = str(Seq(sequence).complement())

    #codon numbers in the FULL protein, 1-based: what the exon's first and last
    #residues are called in the gene, so the pasted fragment can be placed back
    first_aa = cds_index[kept[0]] // 3 + 1
    last_aa = cds_index[kept[-1]] // 3 + 1

    offset = kept.index(genomic_position)
    mutated_aa = cds_index[genomic_position] // 3 + 1
    flip = (lambda b: str(Seq(b).reverse_complement())) if ann['strand'] == '-' else (lambda b: b)
    ref_tx = flip(ref) if ref else None
    alt_tx = flip(alt) if alt else None

    protein = str(Seq(sequence).translate())

    if verbose:
        print('%s  %s  %s strand   exon %d of %d, chr%s:%d-%d'
              % (gene, tx, ann['strand'], exon_index + 1, len(ann['cds']),
                 str(ann['chrom']).replace('chr', ''), exon[0], exon[1]))
        print('%d nt (%d trimmed to codon boundaries), %d codons'
              % (len(sequence), len(span) - len(kept), len(sequence) // 3))
        print('amino acids %d-%d of %s (%s%d ... %s%d)'
              % (first_aa, last_aa, gene,
                 protein[0], first_aa, protein[-1], last_aa))
        print()
        print('paste the sequence; mutation at position %d (1-based), '
              'reference base %r' % (offset + 1, sequence[offset]))
        print('  that is %s%d in the full protein (codon %d of this exon)'
              % (protein[offset // 3], mutated_aa, offset // 3 + 1))
        if ref_tx:
            print('  expected %r on the transcript strand: %s' % (
                ref_tx, 'matches' if sequence[offset] == ref_tx
                else 'DOES NOT MATCH -- wrong build or coordinate?'))
        if alt_tx:
            print('  type %r into "Replace with"' % alt_tx)
        print('  room for bystanders in this exon: %d nt 5\', %d nt 3\''
              % (offset, len(sequence) - offset - 1))

    return {'gene': gene, 'transcript_id': tx, 'strand': ann['strand'],
            'protein_change': protein_call, 'genomic_position': genomic_position,
            'chrom': ann['chrom'], 'exon_index': exon_index + 1,
            'exon_block': exon, 'sequence': sequence, 'protein': protein,
            'first_aa': first_aa, 'last_aa': last_aa,
            'position': offset + 1, 'ref': sequence[offset],
            'mutated_aa': mutated_aa,
            'ref_transcript': ref_tx, 'alt_transcript': alt_tx}

In [ ]:
exon = get_exon('Dnmt3a', 'R878H',species='mouse', build=39)

R878H  ->  codon 878 CGC>CAC at chr12:3957653 (G>A on the genome)

Dnmt3a  ENSMUST00000020991.15  + strand   exon 22 of 22, chr12:3957606-3957744
138 nt (1 trimmed to codon boundaries), 46 codons
amino acids 863-908 of Dnmt3a (V863 ... V908)

paste the sequence; mutation at position 47 (1-based), reference base 'G'
  that is R878 in the full protein (codon 16 of this exon)
  expected 'G' on the transcript strand: matches
  type 'A' into "Replace with"
  room for bystanders in this exon: 46 nt 5', 91 nt 3'


In [7]:
mouse = get_exon('Flt3', 147291714, species='mouse', build=39)

Flt3  ENSMUST00000049324.13  - strand   exon 11 of 24, chr5:147291609-147291741
132 nt (1 trimmed to codon boundaries), 44 codons
amino acids 570-613 of Flt3 (Q570 ... F613)

paste the sequence; mutation at position 28 (1-based), reference base 'A'
  that is M579 in the full protein (codon 10 of this exon)
  room for bystanders in this exon: 27 nt 5', 104 nt 3'


In [8]:
mouse['sequence']

'CAATTTAGGTACGAGAGTCAGCTGCAGATGATCCAGGTGACTGGCCCCCTGGATAACGAGTACTTCTACGTTGACTTCAGGGACTATGAATATGACCTTAAGTGGGAGTTCCCGAGAGAGAACTTAGAGTTT'